# Recursive Methods


```{contents}
:local:
:depth: 2
```


This supporting reading extends the chapter's method skills; it is not a prerequisite for the required assignments. Read it once you are comfortable defining a method, passing arguments, and returning a value. For related explanation, see the further reading below.[^development-reading]


```{index} A Method May Call Itself
```

## A Method May Call Itself


Every method so far has called *other* methods. A method may also call itself, which is called **recursion**. That sounds circular, but it is the same mechanism as any other call: the caller pauses, the called method runs with its own parameters and local variables, and the result comes back. What makes recursion work is that each call handles a *smaller* version of the task, until one call is small enough to answer directly.

A recursive method therefore has two parts:

```csharp
static ReturnType Name(parameters)
{
    if (the input is the smallest case)
        return the direct answer;                     // base case
    return combine(Name(a smaller input));           // recursive case
}
```

- The **base case** is an input the method answers without calling itself.
- The **recursive case** calls the method on a smaller input and uses that result.
- Every recursive call must move toward the base case, or the calls never end.

The business examples below are small on purpose. Recursion earns its keep later, on data that is itself nested, such as folders inside folders or an organization chart (Chapter 17), and in algorithms that split work in half (Chapter 22).


```{index} Trace the Calls: A Deadline Countdown
```

## Trace the Calls: A Deadline Countdown


A reminder service prints how many days remain before a deadline, one line per day, then a final notice. A loop would do this; the recursive version shows the two parts clearly. The base case is zero days left. The recursive case prints the current count and asks the method to handle one day fewer.


In [ ]:
using System;

Countdown(3);

static void Countdown(int daysLeft)
{
    if (daysLeft == 0)
    {
        Console.WriteLine("Deadline today.");
        return;                      // base case: nothing smaller to do
    }
    Console.WriteLine($"{daysLeft} day(s) left.");
    Countdown(daysLeft - 1);         // recursive case: one day fewer
}


Read the trace as four separate calls, each with its own `daysLeft`:

| Call | `daysLeft` | Prints | Then |
| --- | ---: | --- | --- |
| `Countdown(3)` | 3 | 3 day(s) left. | calls `Countdown(2)` |
| `Countdown(2)` | 2 | 2 day(s) left. | calls `Countdown(1)` |
| `Countdown(1)` | 1 | 1 day(s) left. | calls `Countdown(0)` |
| `Countdown(0)` | 0 | Deadline today. | returns |

Each call decreases `daysLeft` by one, so a call with a nonnegative start always reaches zero. A negative argument would never reach the base case; a complete contract either rejects negative input or uses `daysLeft <= 0` as the base condition.


```{index} Guided Practice: Count Up Instead
```

### Guided Practice: Count Up Instead


Make the method print `1 2 3` instead of `3 2 1` by moving one statement. Nothing else changes: the base case and the recursive call stay where they are.


In [ ]:
using System;

CountUp(3);
Console.WriteLine();

static void CountUp(int n)
{
    if (n == 0) return;
    // Your code starts here.
    Console.Write(n + " ");
    CountUp(n - 1);
    // Your code ends here.
}


In [ ]:
using System;

// Solution
CountUp(3);
Console.WriteLine();

static void CountUp(int n)
{
    if (n == 0) return;
    CountUp(n - 1);           // first let the smaller calls print
    Console.Write(n + " ");  // then print this call's number
}


Printing after the recursive call means the deepest call, `CountUp(1)`, prints first, because every outer call is still waiting on the stack for its inner call to return. Work done before the call happens on the way down; work done after it happens on the way back up.


```{index} Return a Value: Sum the First n Numbers
```

## Return a Value: Sum the First n Numbers


Value-returning recursion builds its answer from the smaller call's answer. The sum of the numbers 1 through `n` is `n` plus the sum of the numbers 1 through `n - 1`, and the sum through 0 is 0. The loop version from Chapter 4 computes the same total; compare the two.


In [ ]:
using System;

Console.WriteLine($"Recursive: {SumToRecursive(4)}");
Console.WriteLine($"Loop:      {SumToLoop(4)}");
Console.WriteLine($"Zero:      {SumToRecursive(0)}");

static int SumToRecursive(int n)
{
    if (n <= 0) return 0;                 // base case
    return n + SumToRecursive(n - 1);     // recursive case
}

static int SumToLoop(int n)
{
    int total = 0;
    for (int i = 1; i <= n; i++) total += i;
    return total;
}


Why trust the recursive version? Check the base case directly: `SumToRecursive(0)` is 0. Then assume the smaller call is right: if `SumToRecursive(3)` returns 6, then `4 + 6` is the correct sum through 4. That two-step argument, base case plus smaller case, is how you check any recursive method, and it is the same reasoning Chapter 17 uses for trees.


```{index} The Call Stack and Why Progress Matters
```

## The Call Stack and Why Progress Matters


While `SumToRecursive(4)` waits for `SumToRecursive(3)`, its own `n` is still 4. The runtime keeps every unfinished call on the **call stack**, each with its own parameters, and finishes them in reverse order. The following version prints when each call starts and what it returns, with indentation showing how deep the stack is.


In [ ]:
using System;

Console.WriteLine($"Result: {SumTo(3, "")}");

static int SumTo(int n, string indent)
{
    Console.WriteLine($"{indent}enter SumTo({n})");
    if (n <= 0)
    {
        Console.WriteLine($"{indent}return 0");
        return 0;
    }
    int result = n + SumTo(n - 1, indent + "  ");
    Console.WriteLine($"{indent}return {result}");
    return result;
}


Four calls are open at the deepest point. Each one holds a little memory until it returns, which is why a recursive method that never reaches its base case fails: the stack fills up and the program stops with a `StackOverflowException` that cannot be caught. Forgetting the base case, or calling the method with an input that is not smaller, are the two mistakes to check first. For the same reason, keep teaching examples to small inputs; `SumToRecursive(1000000)` would open a million frames where a loop needs one variable.


```{index} Recursion or a Loop?
```

## Recursion or a Loop?


Both forms can express the same computation. Choose by clarity:

| Prefer a loop when | Prefer recursion when |
| --- | --- |
| The task repeats a step a known number of times | The task is naturally defined in terms of a smaller task |
| Only a running total or counter must be tracked | Each step must remember where it was before going deeper |
| Inputs can be large | The data itself is nested, such as folders or org charts |

Chapter 4's greatest common divisor is a good example of a method whose recursive form reads almost exactly like its mathematical definition: the GCD of `a` and `b` is `a` when `b` is zero, and otherwise the GCD of `b` and the remainder `a % b`.


In [ ]:
using System;

Console.WriteLine($"GCD of 48 and 18 (loop):      {GcdLoop(48, 18)}");
Console.WriteLine($"GCD of 48 and 18 (recursive): {GcdRecursive(48, 18)}");

static int GcdLoop(int a, int b)
{
    while (b != 0)
    {
        int remainder = a % b;
        a = b;
        b = remainder;
    }
    return a;
}

static int GcdRecursive(int a, int b)
{
    if (b == 0) return a;               // base case
    return GcdRecursive(b, a % b);      // smaller: the remainder is less than b
}


The recursive call's second argument, `a % b`, is always smaller than `b`, so the calls make progress toward the base case. Stating that progress measure is part of writing a correct recursive method, just as stating a loop's exit condition is part of writing a correct loop.


```{index} Guided Practice: Count the Digits of an Order Number
```

## Guided Practice: Count the Digits of an Order Number


A report pads order numbers to a fixed width, so it needs to know how many digits a number has. Complete `CountDigits` recursively for nonnegative input: a number below 10 has one digit; otherwise it has one more digit than the number with its last digit removed, `orderNumber / 10`. Expected output: `1`, `1`, `5`.


In [ ]:
using System;

Console.WriteLine(CountDigits(0));
Console.WriteLine(CountDigits(7));
Console.WriteLine(CountDigits(20473));

static int CountDigits(int orderNumber)
{
    // Your code starts here.
    return 0;
    // Your code ends here.
}


In [ ]:
using System;

// Solution
Console.WriteLine(CountDigits(0));
Console.WriteLine(CountDigits(7));
Console.WriteLine(CountDigits(20473));

static int CountDigits(int orderNumber)
{
    if (orderNumber < 10) return 1;            // base case: one digit
    return 1 + CountDigits(orderNumber / 10);  // recursive case: drop the last digit
}


Check the solution the same way as before: the base case is right for 0 through 9, and if the method is right for 2047 (four digits) then it is right for 20473 (one more). Then write the loop version and decide which one you would rather maintain.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Harrington, Recursion](https://introcs.cs.luc.edu/recursion/index.html) and [Nakov and colleagues, Recursion](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-10-recursion/). The reminder and order-number scenarios here are original teaching examples.
